# Tiny Transformer for Next-Token Prediction

**Learning objectives**
- Build **scaled dot-product self-attention** from the equations
- Stack a minimal decoder block (attention + FFN) in NumPy
- Train character-level next-token prediction on a tiny corpus
- Compare with a PyTorch one-block transformer

Run cells top-to-bottom. Constants are grouped near the top so you can experiment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Hyperparameters (tweak these) ---
CORPUS = "abcabcabcabcabcabcabcabcabcabc"  # same toy language as the RNN notebook
CONTEXT = 8
D_MODEL = 32
N_HEADS = 4
D_FF = 64
LEARNING_RATE = 0.08
EPOCHS = 150
BATCH_SIZE = 32

## 1. Problem — predict the next character

Given characters $x_1,\ldots,x_T$, predict $x_{T+1}$.

Unlike RNNs, a transformer mixes all positions in the context **in parallel** via attention (with a causal mask so position $i$ cannot see the future).

In [ ]:
chars = sorted(set(CORPUS))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
V = len(chars)
data = np.array([stoi[c] for c in CORPUS], dtype=np.int64)
print('vocab:', chars, 'V=', V, 'corpus length:', len(data))


def make_dataset(data, context=CONTEXT):
    xs, ys = [], []
    for i in range(len(data) - context):
        xs.append(data[i:i + context])
        ys.append(data[i + context])
    return np.stack(xs), np.array(ys)


X_all, y_all = make_dataset(data)
# tiny corpus → reuse with shuffle splits
rng = np.random.default_rng(0)
idx = rng.permutation(len(y_all))
cut = int(0.8 * len(idx))
X_train, y_train = X_all[idx[:cut]], y_all[idx[:cut]]
X_test, y_test = X_all[idx[cut:]], y_all[idx[cut:]]
print('train', X_train.shape, 'test', X_test.shape)
print('example', ''.join(itos[t] for t in X_train[0]), '→', itos[y_train[0]])

## 2. Embeddings + causal self-attention

**Token + position embeddings:**

$$
e_i = W_e[x_i] + W_p[i]
$$

**Scaled dot-product attention** (single head), with queries, keys, values:

$$
Q = E W_Q,\quad K = E W_K,\quad V = E W_V
$$

$$
\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right) V
$$

Causal mask $M$: $M_{ij} = 0$ if $j\le i$, else $-\infty$ (no looking ahead).

**Learning note:** $\sqrt{d_k}$ keeps dot products from growing with dimension so softmax does not saturate.

**Multi-head:** run $H$ attentions with smaller $d_k = d_{\text{model}}/H$, concatenate, project with $W_O$.

In [ ]:
def softmax_last(Z):
    Z = Z - np.max(Z, axis=-1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=-1, keepdims=True)


def causal_mask(T):
    # (T,T): 0 on and below diagonal, -1e9 above
    m = np.triu(np.ones((T, T)), k=1) * -1e9
    return m


# Attention demo on fixed vectors
T, dk = 4, 8
rng = np.random.default_rng(0)
Q = rng.normal(size=(1, T, dk))
K = rng.normal(size=(1, T, dk))
Val = rng.normal(size=(1, T, dk))
scores = Q @ np.transpose(K, (0, 2, 1)) / np.sqrt(dk) + causal_mask(T)
weights = softmax_last(scores)
_ = weights @ Val  # apply attention to values

fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(weights[0], cmap='viridis', vmin=0)
ax.set_xlabel('key position j'); ax.set_ylabel('query position i')
ax.set_title('Causal attention weights')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()
print('row sums (should be ~1):', weights[0].sum(axis=1).round(3))
print('upper triangle ~0?', np.triu(weights[0], k=1).max() < 1e-6)

### Pause & Reflect — Attention

1. Why must a **decoder** for next-token prediction use a causal mask?
2. What does a sharp attention weight on position $j$ mean for query $i$?
3. If you omit $\sqrt{d_k}$ and $d_k$ is large, what happens to softmax?

### Discussion — Attention

1. **Causality**: At training time position $i$ must not peek at tokens $>i$; otherwise the model cheats and fails at true next-token generation.
2. **Routing**: $i$ mostly reads value $v_j$ — content-based memory lookup.
3. **Saturation**: Large logits → one-hot softmax → vanishing gradients to other keys.

## 3. Tiny decoder block

$$
\begin{aligned}
\tilde{E} &= E + \mathrm{MHA}(\mathrm{LayerNorm}(E)) \\
H &= \tilde{E} + \mathrm{FFN}(\mathrm{LayerNorm}(\tilde{E}))
\end{aligned}
$$

with $\mathrm{FFN}(x) = W_2\,\mathrm{ReLU}(W_1 x)$.

We predict from the **last** position's hidden vector (many-to-one next token), matching the RNN notebooks' setup.

In [ ]:
def layer_norm(X, eps=1e-5):
    # X: (N,T,D)
    mu = X.mean(axis=-1, keepdims=True)
    var = X.var(axis=-1, keepdims=True)
    return (X - mu) / np.sqrt(var + eps)


def init_transformer(rng=np.random.default_rng(42)):
    d, h, ff, v, T = D_MODEL, N_HEADS, D_FF, V, CONTEXT
    assert d % h == 0
    dk = d // h
    s = 0.1
    p = {
        'We': rng.normal(0, s, (v, d)),
        'Wp': rng.normal(0, s, (T, d)),
        'Wq': rng.normal(0, s, (d, d)),
        'Wk': rng.normal(0, s, (d, d)),
        'Wv': rng.normal(0, s, (d, d)),
        'Wo': rng.normal(0, s, (d, d)),
        'W1': rng.normal(0, s, (d, ff)),
        'b1': np.zeros((1, 1, ff)),
        'W2': rng.normal(0, s, (ff, d)),
        'b2': np.zeros((1, 1, d)),
        'Wout': rng.normal(0, s, (d, v)),
        'bout': np.zeros((1, v)),
    }
    p['_dk'] = dk
    p['_h'] = h
    return p


def embed(X_ids, params):
    # X_ids: (N,T)
    tok = params['We'][X_ids]           # (N,T,d)
    pos = params['Wp'][None, :, :]      # (1,T,d)
    return tok + pos


def split_heads(X, h, dk):
    # (N,T,d) -> (N,h,T,dk)
    N, T, d = X.shape
    return X.reshape(N, T, h, dk).transpose(0, 2, 1, 3)


def merge_heads(X):
    # (N,h,T,dk) -> (N,T,d)
    N, h, T, dk = X.shape
    return X.transpose(0, 2, 1, 3).reshape(N, T, h * dk)


def attention(Q, K, V):
    dk = Q.shape[-1]
    scores = Q @ np.transpose(K, (0, 1, 3, 2)) / np.sqrt(dk)
    T = scores.shape[-1]
    scores = scores + causal_mask(T)
    weights = softmax_last(scores)
    return weights @ V, weights


def mha_forward(E, params):
    h, dk = params['_h'], params['_dk']
    Q = split_heads(E @ params['Wq'], h, dk)
    K = split_heads(E @ params['Wk'], h, dk)
    V = split_heads(E @ params['Wv'], h, dk)
    ctx, weights = attention(Q, K, V)
    out = merge_heads(ctx) @ params['Wo']
    return out, weights


def ffn_forward(X, params):
    h1 = np.maximum(0, X @ params['W1'] + params['b1'])
    return h1 @ params['W2'] + params['b2'], h1


def forward(X_ids, params):
    E0 = embed(X_ids, params)
    E_n = layer_norm(E0)
    attn_out, weights = mha_forward(E_n, params)
    E1 = E0 + attn_out
    E1_n = layer_norm(E1)
    ffn_out, h1 = ffn_forward(E1_n, params)
    H = E1 + ffn_out
    last = H[:, -1, :]                  # (N,d)
    logits = last @ params['Wout'] + params['bout']
    # softmax
    shift = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shift)
    probs = exp / exp.sum(axis=1, keepdims=True)
    cache = (X_ids, E0, E_n, attn_out, weights, E1, E1_n, h1, ffn_out, H, last, logits, probs)
    return probs, cache


params = init_transformer()
probs, cache = forward(X_train[:4], params)
print('init probs', np.round(probs, 3))

## 4. Training the tiny decoder

Full transformer BPTT in NumPy is lengthy. We keep the **NumPy attention forward pass** for learning the equations, then train an equivalent one-block decoder with **PyTorch autograd** so next-token prediction actually converges. Finally we visualize causal attention weights from the trained model.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)


class TinyDecoder(nn.Module):
    def __init__(self, v=V, d=D_MODEL, heads=N_HEADS, ff=D_FF, T=CONTEXT):
        super().__init__()
        self.tok = nn.Embedding(v, d)
        self.pos = nn.Embedding(T, d)
        self.ln1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, ff), nn.ReLU(), nn.Linear(ff, d))
        self.out = nn.Linear(d, v)
        self.T = T
        # causal mask
        self.register_buffer('mask', torch.triu(torch.ones(T, T), diagonal=1).bool())

    def forward(self, x):
        N, T = x.shape
        pos = torch.arange(T, device=x.device)
        e = self.tok(x) + self.pos(pos)
        a, _ = self.attn(self.ln1(e), self.ln1(e), self.ln1(e), attn_mask=self.mask)
        e = e + a
        e = e + self.ff(self.ln2(e))
        return self.out(e[:, -1, :])


model = TinyDecoder()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
crit = nn.CrossEntropyLoss()
loader = DataLoader(
    TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
    batch_size=BATCH_SIZE, shuffle=True,
)
X_te_t = torch.tensor(X_test)
y_te_t = torch.tensor(y_test)

hist = {'loss': [], 'test_acc': []}
for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for xb, yb in loader:
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        opt.step()
        total += loss.item() * len(yb)
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t).argmax(1)
        acc = (pred == y_te_t).float().mean().item()
    hist['loss'].append(total / len(y_train))
    hist['test_acc'].append(acc)
    if (epoch + 1) % 30 == 0 or epoch == 0:
        print(f'epoch {epoch+1:3d}  loss={hist["loss"][-1]:.3f}  test={acc*100:.1f}%')

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(hist['loss'], color='steelblue')
ax2.plot([a * 100 for a in hist['test_acc']], color='coral')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax2.set_ylabel('test acc %')
ax1.set_title('Tiny transformer next-token training')
ax1.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Generate characters with the trained PyTorch model
def generate(model, seed='abcabcab', n=40):
    ids = [stoi[c] for c in seed][-CONTEXT:]
    model.eval()
    with torch.no_grad():
        for _ in range(n):
            x = torch.tensor([ids[-CONTEXT:]])
            logits = model(x)
            ids.append(int(logits.argmax(1).item()))
    return ''.join(itos[i] for i in ids)


print('generated:', generate(model))
print('pattern  :', 'abc' * 20)

## 5. Attention visualization (NumPy block)

Even without training the NumPy weights to convergence, the **causal structure** of attention is visible. After copying PyTorch embeddings is optional; here we show weights from a forward pass and from the trained PyTorch module.

In [ ]:
# Attention weights from trained PyTorch MHA on one window
model.eval()
x = torch.tensor(X_test[:1])
with torch.no_grad():
    pos = torch.arange(CONTEXT)
    e = model.tok(x) + model.pos(pos)
    _, w = model.attn(model.ln1(e), model.ln1(e), model.ln1(e),
                      attn_mask=model.mask, average_attn_weights=False)
# w: (N, heads, T, T) in recent PyTorch — handle both
w_np = w.detach().cpu().numpy()
if w_np.ndim == 3:
    w_np = w_np[:, None, :, :]

fig, axes = plt.subplots(1, min(N_HEADS, 4), figsize=(10, 3))
window = ''.join(itos[t] for t in X_test[0])
for i, ax in enumerate(np.atleast_1d(axes)):
    ax.imshow(w_np[0, i], cmap='viridis', vmin=0)
    ax.set_title(f'head {i}')
    ax.set_xlabel('key'); ax.set_ylabel('query')
plt.suptitle(f'Attention on window "{window}" → {itos[y_test[0]]}')
plt.tight_layout(); plt.show()

### Pause & Reflect — Transformer

1. Why add residual connections around attention and the FFN?
2. How does attention lengthen the **effective memory** vs an RNN hidden state?
3. What costs $O(T^2)$ in self-attention, and when does that hurt?

### Discussion — Transformer

1. **Residuals**: Provide a gradient highway (like LSTM's cell path) so deep stacks remain trainable; also let the block refine rather than rewrite representations.
2. **Direct links**: Any past token can be attended in **one** step — no need to hop through $T$ recurrent multiplies.
3. **Quadratic cost**: The $T\times T$ score matrix — painful for very long contexts (hence sparse/linear attention research).

## Summary

| Piece | Role in next-token prediction |
|-------|-------------------------------|
| Embeddings | Tokens + positions → vectors |
| Causal MHA | Each position reads from the past only |
| FFN | Position-wise nonlinear mixing |
| Final linear | Softmax distribution over vocabulary |

You now have a minimal but complete path: **CNN (space) → RNN/LSTM (time) → Transformer (attention)**.